# Import libraries and load data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def show_step(df, step_name):
    print(f"\n===== {step_name} =====")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(3))  

data_path = Path("../data_processed/processed_final_data.csv")
df = pd.read_csv(data_path)

show_step(df, "Raw processed_final")


===== Raw processed_final =====
Shape: (653938, 12)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon
0,2021-4D9Y98,Unknown,"เขตลาดพร้าว,การไฟฟ้านครหลวง เขตนวลจันทร์","100.59131,13.80910",กรุงเทพมหานคร,2021-12-13 05:53:36.861064+00:00,เสร็จสิ้น,-1.0,0.0,2023-03-14 12:09:14.947437+00:00,13.80910,100.59131
1,2021-7K6QA3,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ","100.65617,13.72812",กรุงเทพมหานคร,2021-12-21 23:03:58.450912+00:00,เสร็จสิ้น,2.0,0.0,2022-06-24 06:32:34.671236+00:00,13.72812,100.65617
2,2021-7U9RED,Unknown,เขตดุสิต,"100.50848,13.77832",กรุงเทพมหานคร,2021-12-17 08:46:02.610983+00:00,เสร็จสิ้น,5.0,0.0,2023-05-17 06:11:32.463984+00:00,13.77832,100.50848


# Convert timestamps to datetime

In [2]:
before_cols = set(df.columns)

df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
df["last_activity"] = pd.to_datetime(df["last_activity"], errors="coerce")

after_cols = set(df.columns)

print("Added:", after_cols - before_cols)
print("Removed:", before_cols - after_cols)
show_step(df, "After converting timestamp & last_activity")


Added: set()
Removed: set()

===== After converting timestamp & last_activity =====
Shape: (653938, 12)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon
0,2021-4D9Y98,Unknown,"เขตลาดพร้าว,การไฟฟ้านครหลวง เขตนวลจันทร์","100.59131,13.80910",กรุงเทพมหานคร,2021-12-13 05:53:36.861064+00:00,เสร็จสิ้น,-1.0,0.0,2023-03-14 12:09:14.947437+00:00,13.80910,100.59131
1,2021-7K6QA3,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ","100.65617,13.72812",กรุงเทพมหานคร,2021-12-21 23:03:58.450912+00:00,เสร็จสิ้น,2.0,0.0,2022-06-24 06:32:34.671236+00:00,13.72812,100.65617
2,2021-7U9RED,Unknown,เขตดุสิต,"100.50848,13.77832",กรุงเทพมหานคร,2021-12-17 08:46:02.610983+00:00,เสร็จสิ้น,5.0,0.0,2023-05-17 06:11:32.463984+00:00,13.77832,100.50848


# Compute resolution time in days

In [3]:
before_cols = set(df.columns)

df["resolution_time_days"] = (
    df["last_activity"] - df["timestamp"]
).dt.total_seconds() / (3600 * 24)

after_cols = set(df.columns)
print("Added:", after_cols - before_cols)
print("Removed:", before_cols - after_cols)
show_step(df, "After adding resolution_time_days")


Added: {'resolution_time_days'}
Removed: set()

===== After adding resolution_time_days =====
Shape: (653938, 13)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'resolution_time_days']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,resolution_time_days
0,2021-4D9Y98,Unknown,"เขตลาดพร้าว,การไฟฟ้านครหลวง เขตนวลจันทร์","100.59131,13.80910",กรุงเทพมหานคร,2021-12-13 05:53:36.861064+00:00,เสร็จสิ้น,-1.0,0.0,2023-03-14 12:09:14.947437+00:00,13.80910,100.59131,456.260857
1,2021-7K6QA3,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ","100.65617,13.72812",กรุงเทพมหานคร,2021-12-21 23:03:58.450912+00:00,เสร็จสิ้น,2.0,0.0,2022-06-24 06:32:34.671236+00:00,13.72812,100.65617,184.311530
2,2021-7U9RED,Unknown,เขตดุสิต,"100.50848,13.77832",กรุงเทพมหานคร,2021-12-17 08:46:02.610983+00:00,เสร็จสิ้น,5.0,0.0,2023-05-17 06:11:32.463984+00:00,13.77832,100.50848,515.892707


# Create efficiency label (efficient_flag)

In [4]:
before_cols = set(df.columns)

DONE_STATE = "เสร็จสิ้น"
df["is_done"] = df["state"] == DONE_STATE

df["efficient_flag"] = np.where(
    (df["is_done"]) & (df["resolution_time_days"] <= 7),
    1,
    0
)

after_cols = set(df.columns)
print("Added:", after_cols - before_cols)
print("Removed:", before_cols - after_cols)
show_step(df, "After creating is_done & efficient_flag")

print("\nClass balance (efficient_flag):")
print(df["efficient_flag"].value_counts(normalize=True))


Added: {'efficient_flag', 'is_done'}
Removed: set()

===== After creating is_done & efficient_flag =====
Shape: (653938, 15)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'resolution_time_days', 'is_done', 'efficient_flag']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,resolution_time_days,is_done,efficient_flag
0,2021-4D9Y98,Unknown,"เขตลาดพร้าว,การไฟฟ้านครหลวง เขตนวลจันทร์","100.59131,13.80910",กรุงเทพมหานคร,2021-12-13 05:53:36.861064+00:00,เสร็จสิ้น,-1.0,0.0,2023-03-14 12:09:14.947437+00:00,13.80910,100.59131,456.260857,True,0
1,2021-7K6QA3,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ","100.65617,13.72812",กรุงเทพมหานคร,2021-12-21 23:03:58.450912+00:00,เสร็จสิ้น,2.0,0.0,2022-06-24 06:32:34.671236+00:00,13.72812,100.65617,184.311530,True,0
2,2021-7U9RED,Unknown,เขตดุสิต,"100.50848,13.77832",กรุงเทพมหานคร,2021-12-17 08:46:02.610983+00:00,เสร็จสิ้น,5.0,0.0,2023-05-17 06:11:32.463984+00:00,13.77832,100.50848,515.892707,True,0



Class balance (efficient_flag):
efficient_flag
0    0.617337
1    0.382663
Name: proportion, dtype: float64


# Clean up weird / invalid rows

In [5]:
print("Before filtering:", df.shape)

df = df.dropna(subset=["timestamp", "last_activity", "resolution_time_days"])
df = df[(df["resolution_time_days"] >= 0) & (df["resolution_time_days"] <= 365)]

print("After filtering:", df.shape)
show_step(df, "After filtering out invalid resolution times")


Before filtering: (653938, 15)
After filtering: (612229, 15)

===== After filtering out invalid resolution times =====
Shape: (612229, 15)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'resolution_time_days', 'is_done', 'efficient_flag']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,resolution_time_days,is_done,efficient_flag
1,2021-7K6QA3,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ","100.65617,13.72812",กรุงเทพมหานคร,2021-12-21 23:03:58.450912+00:00,เสร็จสิ้น,2.0,0.0,2022-06-24 06:32:34.671236+00:00,13.72812,100.65617,184.311530,True,0
3,2021-7XATFA,{สะพาน},เขตสาทร,"100.52649,13.72060",กรุงเทพมหานคร,2021-09-26 05:03:52.594898+00:00,เสร็จสิ้น,-1.0,0.0,2022-06-06 01:17:12.272904+00:00,13.72060,100.52649,252.842589,True,0
4,2021-8BTWZB,{ท่อระบายน้ำ},"เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.65440,13.68158",กรุงเทพมหานคร,2021-12-22 10:15:33.294829+00:00,เสร็จสิ้น,5.0,0.0,2022-06-20 13:12:04.994440+00:00,13.68158,100.65440,180.122589,True,0


# Add time features

In [6]:
before_cols = set(df.columns)

df["hour"] = df["timestamp"].dt.hour
df["weekday"] = df["timestamp"].dt.weekday  # 0=Mon, 6=Sun
df["month"] = df["timestamp"].dt.month

after_cols = set(df.columns)
print("Added:", after_cols - before_cols)
print("Removed:", before_cols - after_cols)
show_step(df, "After adding hour, weekday, month")


Added: {'month', 'weekday', 'hour'}
Removed: set()

===== After adding hour, weekday, month =====
Shape: (612229, 18)
Columns: ['ticket_id', 'type', 'organization', 'coords', 'province', 'timestamp', 'state', 'star', 'count_reopen', 'last_activity', 'lat', 'lon', 'resolution_time_days', 'is_done', 'efficient_flag', 'hour', 'weekday', 'month']


,ticket_id,type,organization,coords,province,timestamp,state,star,count_reopen,last_activity,lat,lon,resolution_time_days,is_done,efficient_flag,hour,weekday,month
1,2021-7K6QA3,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ","100.65617,13.72812",กรุงเทพมหานคร,2021-12-21 23:03:58.450912+00:00,เสร็จสิ้น,2.0,0.0,2022-06-24 06:32:34.671236+00:00,13.72812,100.65617,184.311530,True,0,23,1,12
3,2021-7XATFA,{สะพาน},เขตสาทร,"100.52649,13.72060",กรุงเทพมหานคร,2021-09-26 05:03:52.594898+00:00,เสร็จสิ้น,-1.0,0.0,2022-06-06 01:17:12.272904+00:00,13.72060,100.52649,252.842589,True,0,5,6,9
4,2021-8BTWZB,{ท่อระบายน้ำ},"เขตประเวศ,ฝ่ายโยธา เขตประเวศ","100.65440,13.68158",กรุงเทพมหานคร,2021-12-22 10:15:33.294829+00:00,เสร็จสิ้น,5.0,0.0,2022-06-20 13:12:04.994440+00:00,13.68158,100.65440,180.122589,True,0,10,2,12


# Handle missing values in key columns

In [7]:
print("Before fillna (categorical sample):")
display(df[["type", "organization", "province"]].head(3))

cat_cols = ["type", "organization", "province"]
for c in cat_cols:
    df[c] = df[c].fillna("Unknown")

print("After fillna (categorical sample):")
display(df[["type", "organization", "province"]].head(3))


num_cols = ["lat", "lon", "hour", "weekday", "month"]
print("Before fillna (numeric summary):")
display(df[num_cols].isna().sum())

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())

print("After fillna (numeric summary):")
display(df[num_cols].isna().sum())


Before fillna (categorical sample):


,type,organization,province
1,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ",กรุงเทพมหานคร
3,{สะพาน},เขตสาทร,กรุงเทพมหานคร
4,{ท่อระบายน้ำ},"เขตประเวศ,ฝ่ายโยธา เขตประเวศ",กรุงเทพมหานคร


After fillna (categorical sample):


,type,organization,province
1,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ",กรุงเทพมหานคร
3,{สะพาน},เขตสาทร,กรุงเทพมหานคร
4,{ท่อระบายน้ำ},"เขตประเวศ,ฝ่ายโยธา เขตประเวศ",กรุงเทพมหานคร


Before fillna (numeric summary):


lat        0
lon        0
hour       0
weekday    0
month      0
dtype: int64

After fillna (numeric summary):


lat        0
lon        0
hour       0
weekday    0
month      0
dtype: int64

# Choose feature columns

In [8]:
feature_cols = [
    "type",
    "organization",
    "province",
    "hour",
    "weekday",
    "month",
    "lat",
    "lon",
]

target_col = "efficient_flag"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)
display(X.head(3))
display(y.head(3))


X shape: (612229, 8)
y shape: (612229,)


,type,organization,province,hour,weekday,month,lat,lon
1,Unknown,"เขตประเวศ,ฝ่ายเทศกิจ เขตประเวศ",กรุงเทพมหานคร,23,1,12,13.72812,100.65617
3,{สะพาน},เขตสาทร,กรุงเทพมหานคร,5,6,9,13.72060,100.52649
4,{ท่อระบายน้ำ},"เขตประเวศ,ฝ่ายโยธา เขตประเวศ",กรุงเทพมหานคร,10,2,12,13.68158,100.65440


1    0
3    0
4    0
Name: efficient_flag, dtype: int64

# Save features and target for the ML notebook

In [9]:
import os
os.makedirs("../data_processed", exist_ok=True)

X.to_csv("../data_processed/X_features.csv", index=False)
y.to_csv("../data_processed/y_target.csv", index=False)

print("Saved X_features.csv and y_target.csv in ../data_processed")


Saved X_features.csv and y_target.csv in ../data_processed
